In [45]:
# pip install langchain_huggingface 
# !pip install langchain_community

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import re
import contractions
from textblob import TextBlob
import os

## 1. Load the document(.txt)

In [47]:
data = open('data.txt').read()

## 2. Text Normalization

### Converting all the characters into lower case

In [48]:
data = data.lower()

### Removing extra spaces

In [49]:
data = re.sub(r'\s{2,}','',data)

### Removing numbers like 1.,2.

In [50]:
data = re.sub(r'\d+','',data)
data

'. machine learning uses support vector machines to support supply chain optimization while improving loss functions in modern systems.\n. in sentiment analysis, deep learning benefits from k nearest neighbors because it strengthens activation functions and improves decision quality.\n. neural networks often applies naive bayes to document summarization, where data augmentation helps the model generalize more effectively.\n. a practical study of natural language processing shows that principal component analysis can improve video understanding by refining class imbalance during training.\n. researchers use computer vision together with convolutional neural networks to solve personalized tutoring and better understand overfitting control.\n. when reinforcement learning is combined with recurrent neural networks, practitioners can address smart agriculture with more reliable model interpretability.\n. recommendation systems and transformers are frequently paired in traffic prediction, wi

### Contractions

In [51]:
data = contractions.fix(data)

### Removing punctuations & spl characters

In [52]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)

### TextBlob

In [53]:
# values= TextBlob(data).correct()
# values

### Lemmatization

In [54]:
import spacy

nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()
data

'machine learning use support vector machine support supply chain optimization improve loss function modern system \n  sentiment analysis deep learning benefit k near neighbor strengthen activation function improve decision quality \n  neural network apply naive baye document summarization datum augmentation help model generalize effectively \n  practical study natural language processing show principal component analysis improve video understanding refine class imbalance training \n  researcher use computer vision convolutional neural network solve personalized tutoring well understand overfitte control \n  reinforcement learning combine recurrent neural network practitioner address smart agriculture reliable model interpretability \n  recommendation system transformer frequently pair traffic prediction online learning guide learning process \n  cybersecurity monitoring time series forecasting leverage long short term memory feature engineering build adaptable accurate model \n  anoma

### Chunking(Converting doc -> chunks)

In [55]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap = 40
)

chunks = splitter.create_documents([data])
chunks

[Document(metadata={}, page_content='machine learning use support vector machine support supply chain optimization improve loss function modern system'),
 Document(metadata={}, page_content='sentiment analysis deep learning benefit k near neighbor strengthen activation function improve decision quality'),
 Document(metadata={}, page_content='neural network apply naive baye document summarization datum augmentation help model generalize effectively'),
 Document(metadata={}, page_content='practical study natural language processing show principal component analysis improve video understanding refine class imbalance training'),
 Document(metadata={}, page_content='researcher use computer vision convolutional neural network solve personalized tutoring well understand overfitte control'),
 Document(metadata={}, page_content='reinforcement learning combine recurrent neural network practitioner address smart agriculture reliable model interpretability'),
 Document(metadata={}, page_content='r

In [ ]:
rel_chunks= {
    'What is Machine Learning ?':[
                                'machine learning use support vector machine support supply chain optimization improve loss function modern system',
                                'recommendation system transformer frequently pair traffic prediction online learning guide learning process',
                                'sentiment analysis reinforcement learning benefit k near neighbor strengthen activation function improve decision quality',
                                ]
}

In [57]:
chunks[0].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learning use support vector machine support supply chain optimization improve loss function modern system'),
 Document(metadata={}, page_content='sentiment analysis deep learning benefit k near neighbor strengthen activation function improve decision quality'),
 Document(metadata={}, page_content='neural network apply naive baye document summarization datum augmentation help model generalize effectively'),
 Document(metadata={}, page_content='practical study natural language processing show principal component analysis improve video understanding refine class imbalance training'),
 Document(metadata={}, page_content='researcher use computer vision convolutional neural network solve personalized tutoring well understand overfitte control'),
 Document(metadata={}, page_content='reinforcement learning combine recurrent neural network practitioner address smart agriculture reliable model interpretability'),
 Document(metad

### Chunk Embeddings (Converts chunks to vectors)

In [58]:
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learning use support vector machine support supply chain optimization improve loss function modern system'),
 Document(metadata={}, page_content='sentiment analysis deep learning benefit k near neighbor strengthen activation function improve decision quality'),
 Document(metadata={}, page_content='neural network apply naive baye document summarization datum augmentation help model generalize effectively'),
 Document(metadata={}, page_content='practical study natural language processing show principal component analysis improve video understanding refine class imbalance training'),
 Document(metadata={}, page_content='researcher use computer vision convolutional neural network solve personalized tutoring well understand overfitte control'),
 Document(metadata={}, page_content='reinforcement learning combine recurrent neural network practitioner address smart agriculture reliable model interpretability'),
 Document(metad

In [59]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-V2'
)

In [60]:
vectordb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [65]:
user_query = 'What is Machine Learning ?'
r_chunks = vectordb.similarity_search(user_query)

In [66]:
r_chunks = {chunk.page_content for chunk in r_chunks}
r_chunks

{'machine learning use support vector machine support supply chain optimization improve loss function modern system'}

In [67]:
# updated_r_chunks = set()

# for chunk in r_chunks:
#     updated_r_chunks.add(chunk.page_content)

# R_text ='\n'.join(updated_r_chunks)
# R_text

In [ ]:
def r_search(query,k=3):
    R_chunks = vectordb.similarity_search(query,k=k)
    # print(R_chunks)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    return R_Text
def g_text(r_search,query):
        import os
        prompt = f'''
                    You're a helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    for this input = > {query}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
                    Output structure:
                    Input : {query}
                    Output : structured output
            '''
        llm_model = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash", #gemini-3.5-flash,gemini-3.6-flash,gemini-2.5-pro
            api_key=os.environ['GEMINI_API_KEY'],
            temperature = 0.0
        )
        response = llm_model.invoke(prompt).content
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

Input : Explain Machine Learning
Output :
1793 machine learning apply policy gradient medical diagnosis backpropagation help model generalize effectively
1633 machine learning apply policy gradient medical diagnosis backpropagation help model generalize effectively
1553 machine learning apply policy gradient medical diagnosis backpropagation help model generalize effectively


In [ ]:
def recall_at_k(retrieved, relevant, k: int) -> float:
    """
    Recall@k = (# relevant docs in top-k) / (# total relevant docs)
    How many of the relevant docs did we actually find?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set #0 
    return len(hits) / len(relevant_set) if relevant_set else 0.0
recall_at_k(list(r_chunks),rel_chunks[user_query],4)

0.3333333333333333

In [ ]:
def precision_at_k(retrieved, relevant, k: int) -> float:
    """
    Precision@k = (# relevant docs in top-k) / k
    How many of our top-k results were actually relevant?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set
    return len(hits) / k
precision_at_k(list(r_chunks),rel_chunks[user_query],4)

0.25

In [73]:
bool([]) #default
bool([1,2,3]) #non default

True

In [74]:
l=[1,2,3]
if l:
    print("1000")
else:
    print("0.0")
    
l=[]
if l:
    print("1000")
else:
    print("0.0")

1000
0.0


In [ ]:
def reciprocal_rank(retrieved, relevant) -> float:
    """
    Reciprocal Rank = 1 / (rank of first relevant doc)
    How quickly did we find the FIRST relevant document?
    """
    relevant_set = set(relevant)
    for rank, retrived_chunk in enumerate(retrieved, start=1):
        if retrived_chunk in relevant_set:
            return 1.0 / rank
    return 0.0  # No relevant doc found
reciprocal_rank(list(r_chunks),rel_chunks[user_query])

1.0

In [ ]:
retrieved = ['c1','c3','c10']
relevant = ['c1','c5','c8']
#retrived_chunk= c1,1
# c1 in relevant
# 1.0/1 = > 1.0


In [ ]:
retrieved = ['c3','c11','c81']
relevant = ['c5','c1','c8'] #1 ,0
#retrived_chunk= c3,1
# c3 in relevant False
# retrived_chunk= c1,2 
# c1 in relevant True
# 1.0/2 = > 0.5

In [ ]:
def reciprocal_rank(retrieved, relevant) -> float:
    """
    Reciprocal Rank = 1 / (rank of first relevant doc)
    How quickly did we find the FIRST relevant document?
    """
    relevant_set = set(relevant)
    retrieved_set = set(retrieved)
    hits = retrieved_set
    return 0.0  # No relevant doc found
reciprocal_rank(list(r_chunks),rel_chunks[user_query])